In [4]:
import sqlite3
from pathlib import Path

import pandas as pd

orders_db_path = Path("../") / "data" / "orders.db"
connection = sqlite3.connect(orders_db_path)

orders_db_path

PosixPath('../data/orders.db')

In [5]:
orders_preview = pd.read_sql_query(
    "SELECT * FROM orders LIMIT 5",
    connection,
)

orders_preview

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
status_counts = pd.read_sql_query(
    """
    SELECT order_status, COUNT(*) AS order_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC, order_status ASC
    """,
    connection,
)

status_counts

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [7]:
monthly_orders = pd.read_sql_query(
    """
    SELECT strftime('%Y-%m', order_purchase_timestamp) AS purchase_month,
           COUNT(*) AS order_count
    FROM orders
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY purchase_month
    ORDER BY purchase_month
    LIMIT 12
    """,
    connection,
)

monthly_orders

,purchase_month,order_count
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


In [8]:
delivery_lag = pd.read_sql_query(
    """
    SELECT order_status,
           AVG(
               julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)
           ) AS avg_days_to_deliver
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
      AND order_purchase_timestamp IS NOT NULL
    GROUP BY order_status
    HAVING avg_days_to_deliver IS NOT NULL
    ORDER BY avg_days_to_deliver DESC
    """,
    connection,
)

delivery_lag

,order_status,avg_days_to_deliver
0,canceled,20.360006
1,delivered,12.558217
